<a href="https://colab.research.google.com/github/nitha604/news-article-categorisation/blob/main/05_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install required libraries (quiet mode)
!pip install scikit-learn pandas numpy -q

In [7]:
# Download spaCy English model
!python -m spacy download en_core_web_sm -q

# ── Imports ──────────────────────────────────────────────────────────────────
import pickle
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import nltk
import spacy

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

print(' All imports done!')


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
 All imports done!


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print(' Google Drive mounted!')


Mounted at /content/drive
 Google Drive mounted!


In [1]:
# ── Load all saved artifacts from Drive ──────────────────────────────────────
import os
import pickle

SAVE_PATH = '/content/drive/MyDrive/news_project/'
MODELS_PATH = '/content/drive/MyDrive/news_project/models/'

def find_file(filename):
    for p in [SAVE_PATH, MODELS_PATH]:
        full = os.path.join(p, filename)
        if os.path.exists(full):
            return full
    raise FileNotFoundError(f"Could not find {filename} in {SAVE_PATH} or {MODELS_PATH}")

with open(find_file('svm_model.pkl'), 'rb') as f:
    model = pickle.load(f)

try:
    vec_path = find_file('vectorizer.pkl')
except FileNotFoundError:
    vec_path = find_file('tfidf_vectorizer.pkl')

with open(vec_path, 'rb') as f:
    vectorizer = pickle.load(f)

with open(find_file('label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

print(' Model loaded successfully!')
print('Categories:', list(le.classes_))


 Model loaded successfully!
Categories: ['Education', 'Politics', 'Sports', 'Technology']


In [2]:
# ── Preprocessing function (self-contained) ──────────────────────────────────
import re
import nltk
import spacy

# Ensure NLTK stopwords are downloaded & imported
try:
    from nltk.corpus import stopwords
    STOP_WORDS = set(stopwords.words('english'))
except (LookupError, NameError):
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOP_WORDS = set(stopwords.words('english'))

# Ensure spaCy model is loaded
try:
    nlp
except NameError:
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        import subprocess
        subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
        nlp = spacy.load('en_core_web_sm')

def preprocess(text):
    """
    Full text preprocessing pipeline:
      1. Lowercase
      2. Remove punctuation & numbers
      3. Remove stopwords
      4. Lemmatize using spaCy
    """
    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 3: Remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS]
    text = ' '.join(tokens)

    # Step 4: Lemmatize using spaCy
    doc = nlp(text)
    lemmatized = [token.lemma_ for token in doc if token.lemma_.strip()]
    return ' '.join(lemmatized)

# Quick sanity check
sample = "India's PM Modi inaugurates the new Parliament building!"
print('Original :', sample)
print('Processed:', preprocess(sample))

Original : India's PM Modi inaugurates the new Parliament building!
Processed: indias pm modi inaugurate new parliament building


In [3]:
# ── Prediction function with emoji confidence bars ────────────────────────────

CATEGORY_EMOJI = {
    'Sports':     '⚽',
    'Education':  '📚',
    'Politics':   '🏛️',
    'Technology': '💻',
}

def make_bar(prob, bar_width=20):
    """Create an emoji progress bar for a probability value."""
    filled = int(round(prob * bar_width))
    empty  = bar_width - filled
    return '█' * filled + '░' * empty


def predict_category(text, verbose=True):
    """
    Predict the news category for a given headline.

    Parameters
    ----------
    text    : str  — raw news headline
    verbose : bool — whether to print the formatted output (default True)

    Returns
    -------
    (predicted_category, confidence_percent)
    """
    # Step 1: Preprocess
    cleaned = preprocess(text)

    # Step 2: TF-IDF vectorization
    X = vectorizer.transform([cleaned])

    # Step 3: Get class probabilities
    proba = model.predict_proba(X)[0]           # shape: (num_classes,)
    categories = list(le.classes_)              # ['Education', 'Politics', 'Sports', 'Technology']

    # Step 4: Identify best prediction
    best_idx        = int(np.argmax(proba))
    predicted_label = categories[best_idx]
    confidence      = proba[best_idx] * 100

    if verbose:
        emoji = CATEGORY_EMOJI.get(predicted_label, '📰')
        print('─' * 55)
        print(f' Input     : "{text}"')
        print(f' Predicted : {emoji} {predicted_label}')
        print(f'Confidence: {confidence:.1f}%')
        print()
        print(' All Probabilities:')
        # Sort by probability descending for readability
        sorted_pairs = sorted(zip(categories, proba), key=lambda x: x[1], reverse=True)
        for cat, p in sorted_pairs:
            bar  = make_bar(p)
            e    = CATEGORY_EMOJI.get(cat, '  ')
            print(f'  {e} {cat:<12}: {bar}  {p*100:.1f}%')
        print('─' * 55)

    return predicted_label, round(confidence, 2)


print('predict_category() is ready to use!')


predict_category() is ready to use!


In [8]:
# ── Test 1: Obvious, clear-cut headlines ─────────────────────────────────────
clear_headlines = [
    # (expected_category, headline)
    ('Sports',     'Virat Kohli scores 150 as India beat Australia in 3rd ODI at Wankhede'),
    ('Education',  'CBSE announces new exam pattern for Class 10 and 12 from 2025'),
    ('Politics',   'Prime Minister Modi inaugurates new Parliament building in New Delhi'),
    ('Technology', 'ISRO successfully launches PSLV-C58 satellite into orbit'),
]

print('=' * 55)
print('         TEST 1 — CLEAR-CUT HEADLINES')
print('=' * 55)

for expected, headline in clear_headlines:
    predicted, conf = predict_category(headline)
    status = ' CORRECT' if predicted == expected else f' WRONG (expected {expected})'
    print(f'  → Result: {status}\n')


         TEST 1 — CLEAR-CUT HEADLINES
───────────────────────────────────────────────────────
 Input     : "Virat Kohli scores 150 as India beat Australia in 3rd ODI at Wankhede"
 Predicted : ⚽ Sports
Confidence: 92.0%

 All Probabilities:
  ⚽ Sports      : ██████████████████░░  92.0%
  📚 Education   : █░░░░░░░░░░░░░░░░░░░  4.1%
  🏛️ Politics    : █░░░░░░░░░░░░░░░░░░░  3.7%
  💻 Technology  : ░░░░░░░░░░░░░░░░░░░░  0.3%
───────────────────────────────────────────────────────
  → Result:  CORRECT

───────────────────────────────────────────────────────
 Input     : "CBSE announces new exam pattern for Class 10 and 12 from 2025"
 Predicted : 📚 Education
Confidence: 97.0%

 All Probabilities:
  📚 Education   : ███████████████████░  97.0%
  💻 Technology  : ░░░░░░░░░░░░░░░░░░░░  2.0%
  🏛️ Politics    : ░░░░░░░░░░░░░░░░░░░░  0.8%
  ⚽ Sports      : ░░░░░░░░░░░░░░░░░░░░  0.2%
───────────────────────────────────────────────────────
  → Result:  CORRECT

───────────────────────────────────────────

In [9]:
# ── Test 2: Ambiguous, tricky headlines ──────────────────────────────────────
tricky_headlines = [
    'Centre allocates ₹1.2 lakh crore for education in Union Budget',    # Politics vs Education
    'IIT Bombay launches free AI course for 1 lakh students on Coursera', # Education vs Technology
    'Sports Minister announces new policy for Olympic athlete funding',    # Politics vs Sports
    "BYJU'S raises funds amid ED investigation",                          # Technology vs Education
]

print('=' * 55)
print('        TEST 2 — TRICKY/AMBIGUOUS HEADLINES')
print('=' * 55)

for headline in tricky_headlines:
    predict_category(headline)
    print()


        TEST 2 — TRICKY/AMBIGUOUS HEADLINES
───────────────────────────────────────────────────────
 Input     : "Centre allocates ₹1.2 lakh crore for education in Union Budget"
 Predicted : 🏛️ Politics
Confidence: 67.7%

 All Probabilities:
  🏛️ Politics    : ██████████████░░░░░░  67.7%
  ⚽ Sports      : █████░░░░░░░░░░░░░░░  27.2%
  💻 Technology  : █░░░░░░░░░░░░░░░░░░░  4.4%
  📚 Education   : ░░░░░░░░░░░░░░░░░░░░  0.6%
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
 Input     : "IIT Bombay launches free AI course for 1 lakh students on Coursera"
 Predicted : 📚 Education
Confidence: 76.6%

 All Probabilities:
  📚 Education   : ███████████████░░░░░  76.6%
  💻 Technology  : ███░░░░░░░░░░░░░░░░░  15.6%
  ⚽ Sports      : █░░░░░░░░░░░░░░░░░░░  7.0%
  🏛️ Politics    : ░░░░░░░░░░░░░░░░░░░░  0.8%
───────────────────────────────────────────────────────

───────────────────────────────────────────────────────
 Input     : "Sports

In [10]:
# ── Batch prediction function ─────────────────────────────────────────────────

def batch_predict(headlines):
    """
    Predict categories for a list of headlines in one go.

    Parameters
    ----------
    headlines : list of str

    Returns
    -------
    pd.DataFrame with columns: Headline, Predicted Category, Confidence (%)
    """
    results = []
    for headline in headlines:
        pred, conf = predict_category(headline, verbose=False)
        emoji = CATEGORY_EMOJI.get(pred, '📰')
        results.append({
            'Headline':            headline,
            'Predicted Category':  f'{emoji} {pred}',
            'Confidence (%)':      conf,
        })
    return pd.DataFrame(results)


# ── 10 mixed test headlines ───────────────────────────────────────────────────
batch_headlines = [
    'Rohit Sharma named captain for T20 World Cup 2024',
    'UGC releases new guidelines for PhD admission 2024',
    'Rahul Gandhi holds rally in Telangana ahead of elections',
    'Google DeepMind releases Gemini 2.0 with multimodal capabilities',
    'MS Dhoni announces retirement from IPL after CSK season',
    'JEE Main 2025 registration begins on NTA official website',
    'Supreme Court issues notice on Manipur violence to Centre',
    'Apple unveils iPhone 17 with satellite calling and AI camera',
    'FIFA World Cup 2026 host cities officially announced',
    'Karnataka government launches free laptop scheme for college students',
]

# Run batch prediction
results_df = batch_predict(batch_headlines)

# Display the table nicely
print(' Batch Prediction Results')
print('=' * 80)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.colheader_justify', 'left')
display(results_df.style
    .set_properties(**{'text-align': 'left'})
    .bar(subset=['Confidence (%)'], color='#90ee90', vmin=0, vmax=100)
    .format({'Confidence (%)': '{:.1f}%'})
    .hide(axis='index')
)

# Summary stats
print(f'\n Average Confidence: {results_df["Confidence (%)"].mean():.1f}%')
print('\n Category Distribution:')
cat_counts = results_df['Predicted Category'].value_counts()
for cat, count in cat_counts.items():
    print(f'  {cat}: {count} headline(s)')


 Batch Prediction Results


Headline,Predicted Category,Confidence (%)
Rohit Sharma named captain for T20 World Cup 2024,⚽ Sports,98.8%
UGC releases new guidelines for PhD admission 2024,💻 Technology,95.3%
Rahul Gandhi holds rally in Telangana ahead of elections,🏛️ Politics,93.3%
Google DeepMind releases Gemini 2.0 with multimodal capabilities,💻 Technology,97.6%
MS Dhoni announces retirement from IPL after CSK season,⚽ Sports,92.9%
JEE Main 2025 registration begins on NTA official website,📚 Education,84.8%
Supreme Court issues notice on Manipur violence to Centre,🏛️ Politics,65.3%
Apple unveils iPhone 17 with satellite calling and AI camera,💻 Technology,89.9%
FIFA World Cup 2026 host cities officially announced,⚽ Sports,97.5%
Karnataka government launches free laptop scheme for college students,💻 Technology,49.6%



 Average Confidence: 86.5%

 Category Distribution:
  💻 Technology: 4 headline(s)
  ⚽ Sports: 3 headline(s)
  🏛️ Politics: 2 headline(s)
  📚 Education: 1 headline(s)
